In [ ]:
from Bio import SeqIO
import random
import csv
import os
import re
import math

In [ ]:
def chunk_sequence(seq, chunk_size=512, stride=256):
    seq = seq.upper()
    for i in range(0, len(seq) - chunk_size + 1, stride):
        yield seq[i:i + chunk_size]


In [ ]:
def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACGTN]", "N", seq)
    return seq

In [ ]:
def fasta_to_csv(fasta_path, csv_writer, label, name, source="GenBank", num_seq=10, min_len=200, max_len=512):
    count = 0
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = clean_sequence(str(record.seq))
        if len(seq) < min_len:
            continue

        seq_id = record.id
        if len(seq) > max_len:
            for chunck_seq in chunk_sequence(seq,max_len,math.ceil(max_len/2)):
                csv_writer.writerow([
                    seq_id,
                    name,
                    chunck_seq,
                    label,
                    source,
                    len(chunck_seq)
                ])
                count += 1
                if count >= num_seq:
                    break
        else :
            csv_writer.writerow([
                seq_id,
                name,
                seq,
                label,
                source,
                len(seq)
            ])
            count += 1
        if count >= num_seq:
            break

    print(f"[✓] {count} séquences enregistrees ")

In [ ]:
def build_dataset(fasta_path, output_csv, isGMO=False, num_seq=10, max_len=512):
    file_exists = os.path.exists(output_csv)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    filename = os.path.basename(fasta_path)
    name = os.path.splitext(filename)[0]
    seqName = "_".join(name.split("_")[:2])

    with open(output_csv, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["id", "name", "sequence", "label", "source", "length"])

        fasta_to_csv(
            fasta_path=fasta_path,
            csv_writer=writer,
            label= 1 if isGMO else 0,
            name=seqName,
            num_seq=num_seq,
            max_len=max_len
        )

    print(f"[✓] Dataset Enregistree vers {output_csv}")

In [38]:
hirsutum_fasta="../raw/Gossypium_hirsutum_v2.1_genomic.fna"
output_csv="../processed/dataset.csv"
build_dataset(fasta_path=hirsutum_fasta, output_csv=output_csv, isGMO=False, max_len=2048, num_seq=5000)

[✓] 5000 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/dataset.csv


In [39]:
unguiculata_fasta="../raw/Vigna_unguiculata_v2_genomic.fna"
output_csv="../processed/dataset.csv"
build_dataset(fasta_path=unguiculata_fasta, output_csv=output_csv, isGMO=False, max_len=2048, num_seq=5000)

[✓] 5000 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/dataset.csv


In [40]:
Oreochromis_niloticus_fasta="../raw/Oreochromis_niloticus_genomic.fna"
output_csv="../processed/splits/val.csv"
build_dataset(fasta_path=Oreochromis_niloticus_fasta, output_csv=output_csv, isGMO=False, max_len=2048, num_seq=2000)

[✓] 2000 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/splits/val.csv


In [41]:
cry1ac_fasta="../raw/bt_cry1Ac_sequences.fasta"
output_csv="../processed/dataset.csv"
build_dataset(fasta_path=cry1ac_fasta, output_csv=output_csv, isGMO=True, max_len=2048)

[✓] 10 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/dataset.csv


In [42]:
cp4epsps_fasta="../raw/bt_cp4epsps_sequences.fasta"
output_csv="../processed/dataset.csv"
build_dataset(fasta_path=cp4epsps_fasta, output_csv=output_csv, isGMO=True, max_len=2048)

[✓] 10 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/dataset.csv


In [43]:
cotton_gmo_fasta="../raw/cotton_gmo.fna"
output_csv="../processed/dataset.csv"
build_dataset(fasta_path=cotton_gmo_fasta, output_csv=output_csv, isGMO=True, max_len=2048)

[✓] 4 séquences enregistrees 
[✓] Dataset Enregistree vers ../processed/dataset.csv


Synthetisation de GMO

In [44]:
def clean(seq):
    seq = seq.upper()
    return re.sub(r"[^ACGT]", "N", seq)

def random_subseq(seq, min_len, max_len):
    L = len(seq)
    size = random.randint(min_len, max_len)
    if L <= size:
        return seq
    start = random.randint(0, L - size)
    return seq[start:start+size]

def mutate(seq, rate=0.001):
    bases = ["A","C","G","T","N"]
    seq = list(seq)
    for i in range(len(seq)):
        if random.random() < rate:
            seq[i] = random.choice(bases)
    return "".join(seq)

In [45]:
def build_synthetic_gmo(plant_genomes, gmo_name, promoter_seqs, gene_seqs, terminator_seqs,
        n_samples=1000, min_flank=50, mutation_rate=0.0005, count=0):

    samples = []
    min_total=1024
    max_total=2048
    attempts = 0
    max_attempts = n_samples * 10

    while len(samples) < n_samples and attempts < max_attempts:
        attempts += 1
        
        plant = random.choice(plant_genomes)
        promoter = random.choice(promoter_seqs)
        gene = random.choice(gene_seqs)
        terminator = random.choice(terminator_seqs)

        core_length = len(promoter) + len(gene) + len(terminator)
        
        
        if core_length + 2 * min_flank > max_total:
            continue  
        
        # Calcule la disponibilité pour les flancs
        min_flank_total = max(2 * min_flank, min_total - core_length)
        max_flank_total = max_total - core_length
        
        if min_flank_total > max_flank_total:
            continue
        
        # Choisi une longueur totale pour les flancs
        total_flank_len = random.randint(min_flank_total, max_flank_total)
        left_flank_len = random.randint(min_flank, total_flank_len - min_flank)
        right_flank_len = total_flank_len - left_flank_len

        left_flank = random_subseq(plant, left_flank_len, left_flank_len)
        right_flank = random_subseq(plant, right_flank_len, right_flank_len)

        synthetic = left_flank + promoter + gene + terminator + right_flank
        synthetic = mutate(synthetic, mutation_rate)
        
        samples.append({
            "id": f"{gmo_name}_{len(samples)+count:06d}",
            "name" : "synthetic_GMO",
            "sequence": synthetic,
            "label": 1,
            "source": "me",
            "length": len(synthetic)
        })

    return samples

In [46]:

def load_fasta(path):
    seqs = []
    for rec in SeqIO.parse(path, "fasta"):
        seqs.append(clean(str(rec.seq)))
    return seqs

In [47]:
def append_to_csv(samples, csv_path):
    file_exists = os.path.exists(csv_path)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["id", "name", "sequence", "label", "source", "length"])

        for s in samples:
            writer.writerow([s["id"], s["name"], s["sequence"], s["label"], s["source"], s["length"]])

In [ ]:
gossypium_genomes = load_fasta("../raw/Gossypium_hirsutum_v2.1_genomic.fna")
gossypium = "Gossypium_hirsutum"
vigna_genomes = load_fasta("../raw/Vigna_unguiculata_v2_genomic.fna")
vigna = "Vigna_unguiculata"

In [ ]:
promoters     = load_fasta("../raw/promoters.fasta")
genes         = load_fasta("../raw/gmo_genes.fasta")
terminators   = load_fasta("../raw/terminators.fasta")

In [ ]:
count = 0
nb_seq = 9976
nb_seq_per = nb_seq//2
while count < nb_seq:
    gossypium_samples = build_synthetic_gmo(plant_genomes=gossypium_genomes, gmo_name=gossypium, promoter_seqs=promoters, gene_seqs=genes,
        terminator_seqs=terminators, n_samples=nb_seq_per, mutation_rate=0.0003, count=count)
    
    vigna_samples = build_synthetic_gmo(plant_genomes=vigna_genomes, gmo_name=vigna, promoter_seqs=promoters, gene_seqs=genes,
        terminator_seqs=terminators, n_samples=nb_seq_per, mutation_rate=0.0003, count=count)
    
    synthetic_samples =  gossypium_samples + vigna_samples
    count += len(synthetic_samples)
    append_to_csv(synthetic_samples, csv_path="../processed/dataset.csv")

print(f"[✓] Generated {count} synthetic GMO sequences")

Melange le jeu de donnees

In [ ]:
import pandas as pd
datash = pd.read_csv("../processed/dataset.csv")
datash = datash.sample(frac=1, random_state=30).reset_index(drop=True)
datash.to_csv("../processed/dataset.csv", index=False)

Subdivision du dataset en train, test et validation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

Train val et test

In [ ]:
def split_dataset(input_csv, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, random_state=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_csv)
    
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Premier decoupage: train et temp (val+test)
    train_df, temp_df = train_test_split(
        df, test_size=(1 - train_ratio), 
        stratify=df["label"], 
        random_state=random_state
    )

    # Second decoupage: val vs test
    val_size = val_ratio / (val_ratio + test_ratio)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_size),
        stratify=temp_df["label"],
        random_state=random_state
    )

    train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir, "val.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

    print("[✓]Dataset split completed:")
    print(f"  Train: {len(train_df)}")
    print(f"  Val:   {len(val_df)}")
    print(f"  Test:  {len(test_df)}")

    print("\nLabel distribution:")
    print("Train:\n", train_df["label"].value_counts(normalize=True))
    print("Val:\n", val_df["label"].value_counts(normalize=True))
    print("Test:\n", test_df["label"].value_counts(normalize=True))

Train et test

In [ ]:
def split_dataset_train_val(input_csv, output_dir, train_ratio=0.8, test_ratio=0.2, random_state=42):
    assert abs(train_ratio + test_ratio - 1.0) < 1e-6

    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_csv)
    
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Premier decoupage: train et val
    train_df, test_df = train_test_split(
        df, test_size=(1 - train_ratio), 
        stratify=df["label"], 
        random_state=random_state
    )


    train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

    print("[✓]Dataset split completed:")
    print(f"  Train: {len(train_df)}")
    print(f"  Test:   {len(test_df)}")

    print("\nLabel distribution:")
    print("Train:\n", train_df["label"].value_counts(normalize=True))
    print("Test:\n", test_df["label"].value_counts(normalize=True))

In [ ]:
split_dataset_train_val(input_csv="../processed/dataset.csv", output_dir="../processed/splits", 
        train_ratio=0.8, val_ratio=0.2, random_state=42)    

In [ ]:
Oreochromis_genomes = load_fasta("../raw/Oreochromis_niloticus_genomic.fna")
Oreochromis = "Oreochromis_niloticus"

In [ ]:
count = 0
nb_seq = 2000
while count < nb_seq:
    Oreochromis_samples = build_synthetic_gmo(plant_genomes=Oreochromis_genomes, gmo_name=Oreochromis, promoter_seqs=promoters, gene_seqs=genes,
        terminator_seqs=terminators, n_samples=nb_seq, mutation_rate=0.0003, count=count)
    
    count += len(Oreochromis_samples)
    append_to_csv(Oreochromis_samples, csv_path="../processed/splits/val.csv")

print(f"[✓] Generated {count} synthetic GMO sequences")

In [ ]:
import pandas as pd
datash = pd.read_csv("../processed/splits/val.csv")
datash = datash.sample(frac=1, random_state=42).reset_index(drop=True)
datash.to_csv("../processed/splits/val.csv", index=False)